<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yhilpisch/colab/blob/main/notebooks/05_cnn_image_classification.ipynb)


# colab — CNN Image Classification on a Free GPU

## _Training a Small CNN on Fashion-MNIST (CPU vs GPU)_

**&copy; Dr. Yves J. Hilpisch**<br>AI-Powered by Different LLMs.

## How to Use This Notebook

- **Goal**: Train a small convolutional neural network on
  Fashion-MNIST and compare CPU vs GPU training speed.
- **Hardware**: Designed for Google Colab (T4 default). GPU is auto-detected.
- **Data**: `torchvision.datasets.FashionMNIST` downloads automatically.
- **GPU Reference**: See `notebooks/colab_gpus.md` for the GPU comparison.

### Roadmap

1. **Environment**: Check GPU and import PyTorch / torchvision.
2. **Data**: Download Fashion-MNIST and visualise sample images.
3. **Model**: Define a compact CNN.
4. **Training**: Run a few epochs on CPU, then on GPU.
5. **Evaluation**: Compare accuracy, runtime, and sample predictions.

### Environment Setup

We verify the GPU and install PyTorch / torchvision if needed.

In [ ]:
#@title Check GPU and install packages
!nvidia-smi
!pip -q install torch torchvision matplotlib

In [ ]:
#@title Imports
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch device: {device}")
print(f"PyTorch version: {torch.__version__}")

### Download and Explore Fashion-MNIST

Fashion-MNIST is a drop-in replacement for MNIST with clothing items.
It downloads automatically and is small enough to train quickly, yet
rich enough to show real learning.

In [ ]:
#@title Load dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_ds = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform,
)
test_ds = torchvision.datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform,
)

BATCH = 256
train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH, shuffle=True, num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=BATCH, shuffle=False, num_workers=2
)

CLASSES = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]
print(f"Train: {len(train_ds):,}, Test: {len(test_ds):,}")

In [ ]:
#@title Visualise sample images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(CLASSES[label])
    ax.axis("off")
plt.tight_layout()
plt.show()

### Define a Compact CNN

Two convolutional blocks with max-pooling followed by a small
fully-connected head. Under 50 k parameters — fast to train on
any hardware.

In [ ]:
#@title Small CNN
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(32 * 7 * 7, 64)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 14x14
        x = self.pool(F.relu(self.conv2(x)))   # 7x7
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SmallCNN()
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total:,}")

### Training Helper

A simple loop that tracks loss and accuracy after every epoch.

In [ ]:
#@title Train / evaluate helper
def train_epoch(model, loader, optim, criterion, dev):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(dev), yb.to(dev)
        optim.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optim.step()
        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(dim=1) == yb).sum().item()
        total += xb.size(0)
    return total_loss / total, correct / total


def evaluate(model, loader, criterion, dev):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(dev), yb.to(dev)
            out = model(xb)
            loss = criterion(out, yb)
            total_loss += loss.item() * xb.size(0)
            correct += (out.argmax(dim=1) == yb).sum().item()
            total += xb.size(0)
    return total_loss / total, correct / total


def fit(model, loader, test_loader, optim, criterion, dev, epochs):
    history = []
    for ep in range(1, epochs + 1):
        t0 = time.perf_counter()
        loss_tr, acc_tr = train_epoch(
            model, loader, optim, criterion, dev
        )
        loss_te, acc_te = evaluate(model, test_loader, criterion, dev)
        dt = time.perf_counter() - t0
        history.append(
            {"epoch": ep, "loss_tr": loss_tr, "acc_tr": acc_tr,
             "loss_te": loss_te, "acc_te": acc_te, "time": dt}
        )
        print(
            f"Ep {ep}/{epochs}  loss={loss_tr:.3f}  "
            f"acc={acc_tr:.3f}  val_loss={loss_te:.3f}  "
            f"val_acc={acc_te:.3f}  ({dt:.1f} s)")
    return history

### CPU Training

Train for 3 epochs on the CPU. With a small model and batch size 256,
this is still manageable — but slow enough to make the GPU speed-up
viscerally obvious.

In [ ]:
#@title Train on CPU
model_cpu = SmallCNN()
opt_cpu = torch.optim.Adam(model_cpu.parameters(), lr=1e-3)
crit = nn.CrossEntropyLoss()

hist_cpu = fit(
    model_cpu, train_loader, test_loader,
    opt_cpu, crit, "cpu", epochs=3,
)
cpu_time = sum(h["time"] for h in hist_cpu)
print(f"\nTotal CPU time: {cpu_time:.1f} s")

### GPU Training

Reset the model weights and train the identical architecture on the
GPU for the same 3 epochs.

In [ ]:
#@title Train on GPU
model_gpu = SmallCNN().to(device)
opt_gpu = torch.optim.Adam(model_gpu.parameters(), lr=1e-3)

hist_gpu = fit(
    model_gpu, train_loader, test_loader,
    opt_gpu, crit, device, epochs=3,
)
torch.cuda.synchronize() if device == "cuda" else None
gpu_time = sum(h["time"] for h in hist_gpu)
print(f"\nTotal GPU time: {gpu_time:.1f} s")

speedup = cpu_time / gpu_time
print(f">>> Training speed-up: {speedup:.1f}×")

### Results and Sample Predictions

Plot training curves and inspect a few test images with predicted
labels from the GPU model.

In [ ]:
#@title Training curves
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

epochs = list(range(1, len(hist_cpu) + 1))
axes[0].plot(epochs, [h["loss_tr"] for h in hist_cpu],
             label="CPU train", marker="o")
axes[0].plot(epochs, [h["loss_tr"] for h in hist_gpu],
             label="GPU train", marker="o")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Train loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs, [h["acc_te"] for h in hist_cpu],
             label="CPU val", marker="o")
axes[1].plot(epochs, [h["acc_te"] for h in hist_gpu],
             label="GPU val", marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
#@title Sample predictions (GPU model)
model_gpu.eval()
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, true_label = test_ds[i]
    with torch.no_grad():
        pred = model_gpu(img.unsqueeze(0).to(device))
    pred_label = pred.argmax(dim=1).item()
    ax.imshow(img.squeeze(), cmap="gray")
    colour = "green" if pred_label == true_label else "red"
    ax.set_title(
        f"{CLASSES[pred_label]}",
        color=colour,
    )
    ax.axis("off")
plt.tight_layout()
plt.show()

### Exercises

1. **More epochs**: Train for 10 epochs on GPU. Does accuracy plateau?
2. **Batch size**: Increase `BATCH` to 512 or 1024. How does GPU
   utilisation change?
3. **CIFAR-10**: Swap Fashion-MNIST for CIFAR-10 (3 channels, 32x32).
   Adjust the first `Conv2d` layer.
4. **Augmentation**: Add `transforms.RandomRotation(10)` to the
   training transform. Does validation accuracy improve?
5. **Larger model**: Add a third conv block (64 channels). How much
   longer does each epoch take on GPU?

<img src="https://theaiengineer.dev/tae_logo_gw_flatter.png" width="35%" align="right">
